# Feature extraction

## Region properties

Once you have segmented an image you usually want to gather information on the objects that you "discovered". Instead of painstakingly do this manually, skimage offers a simplified way to do this with its ```regionprops_table``` tool.

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import skimage
import skimage.io
import skimage.morphology
import skimage.segmentation
import skimage.color
import scipy.ndimage as ndi
import stackview
import ipywidgets as widgets

## Links

* https://scikit-image.org/docs/stable/auto_examples/segmentation/index.html
* https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops

## Download and show the image

In [ ]:
image_stack = skimage.io.imread('images/cells_atlas/46658_784_B12_1.tif')

In [ ]:
plt.subplots(figsize=(5,5))
plt.imshow(image_stack);

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
ax1.imshow(image_stack[:,:,0], cmap='Reds_r')
ax1.set_title('Red')
ax2.imshow(image_stack[:,:,1], cmap='Greens_r')
ax2.set_title('Green')
ax2.axis('off')
ax3.imshow(image_stack[:,:,2], cmap='Blues_r')
ax3.set_title('Blue')
ax3.axis('off')

## Begin processing

Let's first create a mask of the nuclei and clean it up with morphological operations:

In [ ]:


image_nuclei = image_stack[:,:,2] #blue channel in RGB
image_signal = image_stack[:,:,1] #green channel in RGB

# filter image
image_nuclei = skimage.filters.median(image_nuclei, skimage.morphology.disk(5))

# create mask and clean-up
mask_nuclei = image_nuclei > skimage.filters.threshold_otsu(image_nuclei)
mask_nuclei = skimage.morphology.binary_closing(mask_nuclei, footprint=skimage.morphology.disk(5))
mask_nuclei = ndi.binary_fill_holes(mask_nuclei, skimage.morphology.disk(5))

In [ ]:
plt.subplots(figsize=(7,7))
plt.imshow(mask_nuclei, cmap = 'gray');

## Labelling

In order to measure objects in the image separately, we first need to label them individually. For that we can just use the ```skimage.morphology.label()``` function which looks for independent groups of white pixels and assigns them integer numbers:

In [ ]:
my_labels_raw = skimage.morphology.label(mask_nuclei)

In [ ]:
def overlay_labels(label_img, text_color='red'):
    """This is a function not included in sci-kit image. 
    It is for demonstration purposes to show label numbers
    As such, it is not extensively tested!
    This calls a label_image generated from skiimage.mophology.label
    text_color can be used to pick your color"""
    
    img = label_img.astype(float)
    img /= img.max() if img.max() > 0 else 1

    rgb = skimage.color.gray2rgb(img)

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(rgb)

    for region in skimage.measure.regionprops(label_img):
        y, x = region.centroid
        ax.text(x, y, str(region.label),
                color=text_color,
                fontsize=10,
                ha="center", va="center")

    ax.axis("off")
    return fig, ax

In [ ]:
labels = skimage.morphology.label(my_labels_raw)
overlay_labels(labels)
plt.show()


The label map shows that numbers are assigned from top to bottom in the image:

In [ ]:
plt.subplots(figsize=(7,7))
plt.imshow(my_labels_raw);

## Clear Border

We often remove objects that touch the image border, as we can't know what the object is doing on the other side of the border. This step is often done before other image processing, as there will be less objects to process downstream, decreasing processing time. 

In [ ]:
my_labels = skimage.segmentation.clear_border(my_labels_raw)

In [ ]:
plt.subplots(figsize=(7,7))
plt.imshow(my_labels);

## Visualizing Segmentation on Original Image

At times it is useful to preview the segmentation on the original image as a sanity check

In [ ]:
label_overlay = skimage.color.label2rgb(my_labels, image=image_stack[:,:,2], alpha=0.2)
plt.subplots(figsize=(7,7))
plt.imshow(label_overlay);

In [ ]:
label_overlay_u8 = (label_overlay * 255).astype(np.uint8)

In [ ]:
w = stackview.curtain(label_overlay_u8, image_stack[:,:,2])
w.layout = widgets.Layout(width="256px", height="256px")
display(w)


## Region properties

Now that we have each region labeled with a different number we can use the ```skimage.measure.regionprops_table()``` function, which takes such a label map and analyzes some properties of each region. We have to specify which ```properties``` we want to use. 


The list of available ```properties``` can be found in the documentation of the function [here](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops). 

Let's start by adding some of morphological properties in our list of properties and provide some explainations.

- ```label```
The label of the region. It allows us to indentify each segmented object corretly.

- ```area``` and ```perimeter```
Area and perimeter of the region.

Note that the pixel ```spacing``` along each axis of the image can be provided as argument to the function ```skimage.measure.regionprops_table()```. If provided, these properties will be returned in calibrated units. Otherwise, in number of pixels. Even if you did not feed the pixel spacing to the ```skimage.measure.regionprops_table()``` method, you can still convert the results to calibrated units later (as long as you have the spacing information from your image metadata) by a simple multiplication. We will see that later.

Now let's try to call the function with these three properties. 

In [ ]:
my_regions = skimage.measure.regionprops_table(my_labels, properties=('label','area','perimeter'))

The output is a dictionary of all properties that we asked to get out:

In [ ]:
my_regions

### Dictionaries

Until now, in terms of data structures, we have briefly seen lists ```mylist = [5, 4, 2]``` and Numpy arrays via the images. However Python offers additional types of data structures and dictionaries are one of them. As you can see in the output above, they are defined with curly parentheses ```{}``` and contain pairs of elements: keys like ```label``` and ```area``` and a *content* for each key, here two Numpy arrays. To better understand let's just create a simple one:

In [ ]:
my_dict = {'fruit name': 'apple', 'weight': 50, 'types': ['golden', 'gala', 'breaburn']}
my_dict

As you can see these dictionaries can contain all types of variables: strings, numbers, lists etc. They are for this reason ideal to hold information of various types and useful to describe entities thanks to the dictionary keys. Each entry in the dictionary can then be recovered via its key:

In [ ]:
my_dict['weight']

## Recovering image intensity information

In what we did above, we only recovered information about our mask. However often we want to obtain information on pixel values of the **original** image. For example, "what is the average intensity of each nucleus?"

Luckily ```regionprops_table``` allows us to pass as additional argument ```intensity_image``` the image we want to use to quantify intensity. Then we can for example add a property to extract the ```mean_intensity```:

In [ ]:
my_regions = skimage.measure.regionprops_table(
    my_labels,intensity_image=image_signal, properties=('label','area','perimeter','mean_intensity'))

In [ ]:
my_regions

Additionnal properties as the ```intensity_max```, ```intensity_min```, ```intensity_std``` for max, min and std of intensity values in the region, respectively, can also be computed. In some contexts, adding these features and not just looking at mean intensity may be relevant.

Now that we have this information, we can of course, plot it. For example we can produce a histogram of mean nuclei intensities:

In [ ]:
plt.hist(my_regions['mean_intensity']);

## Filtering information

Obviously, we had some "bad segmentations", i.e. some fragments remaining from the processing that are not actual nuclei. We can easily filter those out for example based on size using Numpy logical indexing:

In [ ]:
my_regions['area']

We create a logical array by setting a condition on one dictionary entry:

In [ ]:
selected = my_regions['area'] > 100
selected

And then use it for logical indexing:

In [ ]:
my_regions['mean_intensity'][selected]

## One step further: Pandas

In the above example, if we wanted to use one measurement to filter all other measurements, we would have to repeat the selection multiple times. Ideally, we would put all the measured properties into a table, one column per property, and then do typical database operations to sub-select parts of the data. This can be done using yet another data structure called a DataFrame. These structures are provided by the Pandas library, the main data science library in Python. We here give a very brief insight into that library. First we import it:

In [ ]:
import pandas as pd

To understand what a DataFrame is, let's transform a plain Numpy array into a DataFrame:

In [ ]:
np.random.seed(42)
my_array = np.random.randint(0,100, (3,5))
my_array

We can simply turn this array into a DataFrame by using:

In [ ]:
pd.DataFrame(my_array)

We see that the array content is still there but in addition now we have column and row names, currently just indices. We could however give specific column names:

In [ ]:
my_df = pd.DataFrame(my_array, columns=['a', 'b', 'c', 'd', 'e'])
my_df

The difference with Numpy arrays is that DataFrames can contain different types of information (text, numbers etc.) and that they should really be seen as "organized" data. So for example we can recover a column of the table without resorting to the type of indexing we did before:

In [ ]:
my_df['c']

Now how can such a structure help us to do the sort of data filtering we have mentioned before? Just like with arrays, we can use some constraining tests. For example we can ask: are there data points in column ```c``` which are smaller than 50?

In [ ]:
my_df['c'] < 50

Similarly to what happened with arrays, we get a new column that is boolean. And again similarly to what we did with arrays we can use it for logical indexing using square parentheses:

In [ ]:
my_df[my_df['c'] < 50]

What happened here is that we kept only those entries in the table where the values in the ```c``` column were smaller than 50: we filtered all the properties (columns) in our table in one go!

### Back to our problem

In our analysis we ended up with a dictionary:

In [ ]:
my_regions

We can also easily turn this dictionary into a DataFrame:

In [ ]:
my_regions_df = pd.DataFrame(my_regions)
my_regions_df

And now we can use what we have just learned: let's remove tiny regions with an area smaller than 100:

In [ ]:
my_regions_df[my_regions_df['area'] > 100]

We see that we indeed removed one elements in that table, index 0.

Imagine we forgot to provide the pixel spacing information to ```regionprops_table()``` method but you know this information from your metadata. You can still convert the results to physical units by multiplying the columns concerned by the pixel spacing value as follows:

In [ ]:
pixel_spacing = 0.06 # Let's say that 1 pixel corresponds to 0.06 um in our case
my_regions_df['perimeter um'] = my_regions_df['perimeter'] * pixel_spacing
my_regions_df['area um2'] = my_regions_df['area'] * pixel_spacing**2
my_regions_df

Pandas is a very powerful library and in this course we can't offer more than this brief insight into how it can be useful for data post-processing. To learn more you can also visit this other course: https://guiwitz.github.io/DAVPy/Readme.html

## Excercise 1
Segmentation of blobs and filtering by a morphological feature: We're going to filter out elongated blobs

Steps: 

1. Load blobs and visualize
2. Segment blobs
3. Make measurements to filter the blobs by shape (aspect ratio and eccentricity)
4. Plot a histogram of the aspect ratio or eccentricity
5. Filter the measurements
6. Plot the label image without the elongated shapes. (option: keep original labels)


### 1. Load blobs image from the images folder and vizualise it
   
   Blobs are in "images/blobs.tif"

In [ ]:
# Write your code here




### 2. Segment the blobs
 Threshold the blobs to create a segmentation map.  Create segmentation labels

In [ ]:
# Write your code here




### 3. Find a way to filter the blobs that have elongated shapes.  
Try creating a new dataframe column with a new measure ("aspect_ratio") that would give you a measure of the length to width (maybe the minor axis vs major axis length).  What other measurements might work? Find another measurement that could also work. Filter your blobs by the aspect ratio or other measure

#### Aspect Ratio measure

In [ ]:
# Write your code here




In [ ]:
# Write your code here


#### Eccentricity measure

In [ ]:
# Write your code here


In [ ]:
# Write your code here


### 4. Plot a histogram of the aspect ratio or eccentricity

### 5. Filter the blobs to remove those that are elongated
You can do either aspect ratio and eccentricity. 

In [ ]:
# Write your code here

In [ ]:
# Write your code here

### 6. Plot the label image without the elongated shapes

the np.isin function will compare two groups of elements, and removes from the first group elements that aren't in the second 

https://numpy.org/doc/stable/reference/generated/numpy.isin.html

In [ ]:
# Write your code here

In [ ]:
# Write your code here

### Solution Cell: 

This is the minimum solution for this problem.  It'll help you if you get completely stuck, but it's not easily readible.  Please work on your own first, then check the solutions notebook if you need help

In [ ]:
# This is what we should be getting
#load blobs
image_ex_1 = skimage.io.imread('images/blobs.tif')
#mask and label blobs
mask_blobs = image_ex_1 > skimage.filters.threshold_otsu(image_ex_1)
label_blobs = skimage.morphology.label(mask_blobs)

#measure properties and make aspect ratio
results = skimage.measure.regionprops_table(label_blobs, properties=('label', 'axis_major_length', 'axis_minor_length', 'eccentricity'))
results_df = pd.DataFrame(results)
results_df['aspect_ratio'] = results_df['axis_major_length'] / results_df['axis_minor_length']

# Remove elongated blobs
results_elongated_removed = results_df[results_df['aspect_ratio'] < 1.5]
results_elongated_ecc_removed = results_df[results_df['eccentricity'] < 0.75]

filtered_labels_ecc = np.isin(label_blobs, results_elongated_ecc_removed['label'])
filtered_labels = np.isin(label_blobs, results_elongated_removed['label'])

# Relabel the filtered mask for visualization
filtered_labels_relabel_ecc = skimage.segmentation.relabel_sequential(filtered_labels_ecc * label_blobs)[0]
filtered_labels_relabel = skimage.segmentation.relabel_sequential(filtered_labels * label_blobs)[0]

#visualize
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
ax1.imshow(label_blobs)
ax1.set_title('Blobs')
ax2.imshow(filtered_labels_relabel)
ax2.set_title('Aspect Ratio')
ax2.axis('off')
ax3.imshow(filtered_labels_relabel_ecc)
ax3.set_title('Eccentricity')
ax3.axis('off')

In [ ]:
# Solution cell 2

stackview.curtain(label_blobs, filtered_labels_relabel)

In [ ]:
# Solution cell 3

# Option 2 to keep the same label values as before
filtered_labels = np.where(np.isin(label_blobs, results_elongated_removed['label']), label_blobs, 0)

plt.subplots(figsize=(5,5))
plt.imshow(filtered_labels);

In [ ]:
# Solution cell 4

stackview.curtain(label_blobs, filtered_labels)

# Splitting Objects

Splitting objects is done when we have either elongated objects, or objects that are overlapping/touching.  We often see this with nuclei blobs touching, and they are hard to segment.
Here's the steps to our method of object splitting

1. Distance transformation: For every pixel in our blob label, find the distance to the nearest background pixel.  The peaks in this transform are the "seeds" for our splitting
2. Gaussian smooth our distance transform. Gets rid of noise that would show up as peaks. 
3. define our "footprint" of a 3x3 window to look for local peaks
4. find the local maxima from the smoothed image inside each window (which hopefully is the local maxima inside each blob).  The labels=mask_blobs force the local maxima to by only inside the labels
5. Create a marker image: We start with a blank 'canvas' of background pixels (zeros), and turn them to foreground pixels where there is a local maxima.
6. Run watershed segmentation on the blurred image, using the markers as seed points, and the mask to force the segmentation to stay within the labels.
7. plot it all


In [ ]:
# Solution cell 1

# Step 1 compute distance transform from masks
distance = ndi.distance_transform_edt(mask_blobs)

# Step 2 blur helps reduce the noise and minor irregularities in the distance values 
blurred_distance = skimage.filters.gaussian(distance, sigma=2)

# Step 3 define our foot print
fp = np.ones((3,3))

# Step 4 compute local max of blurred distance map using the footprint
coords = skimage.feature.peak_local_max(blurred_distance, footprint=fp, labels=mask_blobs)

# Step 5 create an array with labeled markers 
mask = np.zeros(distance.shape, dtype=bool)
mask[tuple(coords.T)] = True
markers = skimage.measure.label(mask)

# Step 6 apply the watershed algorithm
labels = skimage.segmentation.watershed(-blurred_distance, markers, mask=mask_blobs)


In [ ]:
#Step 7, plot it all
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1 — mask_blobs
axes[0, 0].imshow(mask_blobs, cmap='gray')
axes[0, 0].set_title("mask_blobs")
axes[0, 0].axis('off')

# 2 — distance transform
axes[0, 1].imshow(distance, cmap='viridis')
axes[0, 1].set_title("distance transform")
axes[0, 1].axis('off')

# 3 — blurred distance
axes[0, 2].imshow(blurred_distance, cmap='viridis')
axes[0, 2].set_title("blurred distance")
axes[0, 2].axis('off')

# 4 — coords (peak local maxima)
coords_img = np.zeros_like(mask_blobs, dtype=float)
coords_img[tuple(coords.T)] = 1
axes[1, 0].imshow(mask_blobs, cmap='gray')
axes[1, 0].scatter(coords[:, 1], coords[:, 0], c='red', s=10)
axes[1, 0].set_title("coords (local maxima)")
axes[1, 0].axis('off')

# 5 — markers
axes[1, 1].imshow(markers, cmap='nipy_spectral')
axes[1, 1].set_title("markers")
axes[1, 1].axis('off')

# 6 — final watershed labels
axes[1, 2].imshow(labels, cmap='nipy_spectral')
axes[1, 2].set_title("watershed labels")
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:


plt.subplots(figsize=(7,7))
plt.imshow(labels)
plt.show()

In [ ]:
overlay_labels(labels)

## Removing pixels between objects split by watershed

#### If you really want to "split" the objects, you can do the following
* Detect the edges from the labels after watershed
* Detect the edges from before watershed
* Use the XOR logical to find the edges created by the watershed
* Grow the watershed edge and remove it
* Remove the watershed edges to split the cells properly

In [ ]:
# first detect edges from the labels after watershed
edges = skimage.filters.sobel(labels)

# then detect edges from the mask_blobs before watershed
edges2 = skimage.filters.sobel(mask_blobs)

fig, ax = plt.subplots(1, 2, figsize=(10,10))
ax[0].imshow(edges)
ax[1].imshow(edges2)
ax[0].set_title('Edges of labels after watershed');
ax[1].set_title('Edges of masks before watershed');

In [ ]:
#Use XOR to remove the edges created by watershed
almost = np.logical_not(np.logical_xor(edges != 0, edges2 != 0)) * mask_blobs # XOR = exclusive OR

#Grow the holes created by removing the boundaries added by watershed
output = skimage.morphology.binary_opening(almost)

#label the new segmentations
split_labels_ex_1 = skimage.morphology.label(output)

fig, ax = plt.subplots(2, 2, figsize=(10,10))
ax[0, 0].imshow(np.logical_xor(edges != 0, edges2 != 0))
ax[0, 0].set_title("Boundaries added by watershed")
ax[0, 1].imshow(almost)
ax[0, 1].set_title("Remove watershed boundary\n note: perimeter is intact")
ax[1, 0].imshow(output)
ax[1, 0].set_title("Split by Watershed")
ax[1, 1].imshow(split_labels_ex_1);
ax[1, 1].set_title("Relabeled after Split by Watershed")

## Computing distances between objects

In some projects, you might be interested in computing distances between different objects. For example, to filter out objects that are far away from an other set of objects. An efficient approach is based on distance maps, and we will see an example of usage along with ```skimage.measure.regionprops_table()``` method in the exercise 2 below. We give before a little theory reminder about distance maps.

Distance transforms have many applications, we can for example use them to quantify how a structure of interest is away from object boundaries or other structures as just mentionned. They are also used to characterize the morphology of an object in 2D and 3D, find its center, dimensions, etc.. Distance transforms can also be used as a pre-processing step to improve the segmentation results and split touching objects. Distance maps may use different distance metrics, as the Euclidian or the Manhattan one for example.

<img src="illustrations/distance_maps.png" alt="drawing" width="50%" class="center"/>

Image [source](https://neubias.github.io/training-resources/distance_transform/index.html#:~:text=We%20use%20distance%20transform%20to,center%2C%20dimensions%2C%20etc..)

## Exercise 2

### Step 1. 
Open hela cells image and visualize the different channels
   
   The hela cells are in the image folder as images\hela-cells.tif

In [ ]:
# Write your code here




### Step 2.
Segment nuclei on the last channel

In [ ]:
# Write your code here




### Step 3.
Detect lysosomes on the first channel and convert them as labels.  Use difference of gaussians. 

In [ ]:
# Write your code here




### Step 4.
Compute distance map and inverse distance map of nuclei. Hint [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.distance_transform_edt.html)

Use stackview.picker to visualize the distances

In [ ]:
# Write your code here




### Step 5.
Compute distances from lysosomes to closest nuclei. Hint: you can use ```regionprops```

In [ ]:
# Write your code here




Note that as for ```skimage.measure.regionprops_table()``` method, you can provide a pixel spacing to the ```scipy.ndimage.
distance_transform_edt()``` method so that the returned distances are in calibrated units.

## [OPTIONAL] Exercise 3

Steps:
1. Import the image from images/4color_cells.tif and visualize the 3rd channel
2. Create a mask using the third channel and label objects
3. Measure the region properties using this labeled image and use the fourth channel as intensity image
4. Extract the mean_intensity property from the dictionary and plot an histogram of these intensities
5. What do you observe in this histogram ? Plot the fourth channel using stackview.picker and see if the histogram makes sense

#### Step 1.
Import the image from images/4color_cells.tif and visualize the 3rd channel

In [ ]:
# Write your code here




### Step 2.
Create a mask using the third channel and label objects

In [ ]:
# Write your code here




### Step 3.
Measure the region properties using this labeled image and use the fourth channel as intensity image
 

In [ ]:
# Write your code here




### Step 4.
Extract the mean_intensity property from the dictionary and plot an histogram of these intensities
 

In [ ]:
# Write your code here




### Step 5.
What do you observe in this histogram ? Plot the fourth channel using stackview.picker and see if the histogram makes sense

In [ ]:
# Write your code here


